# Matrices en 3 colonnes

Repr?sentation d'une matrice avec Spark / Map / Reduce.

Ce notebook propose d'impl?menter un produit matriciel sous Spark. Spark comme SQL n'aime pas trop avoir un nombre de colonnes variables. La premi?re ?tape consiste ? transformer les matrices $I\times J$ en tableau de trois colonnes $(i,j,coefficient)$.

## Session spark with no cluster

Spark est cens? tourner sur un cluster. Mais ce n'est pas essentielle pour comprendre la logique. Le notebook tourne donc en local.

In [ ]:
from pyspark.sql import SparkSession

# Create a SparkSession
spark = SparkSession.builder.appName("MySparkApp").master("local[*]").getOrCreate()
sc = spark.sparkContext

## Cr?ation d'une matrice al?atoire

In [ ]:
from numpy.random import rand

rnd1 = rand(10, 10)
rnd2 = rand(10, 2)
rnd1 @ rnd2

In [ ]:
import pandas

df1 = pandas.DataFrame(rnd1)
df2 = pandas.DataFrame(rnd2)
df2

In [ ]:
df1.to_csv("rnd1.txt", sep="\t", header=None, index=False)
df2.to_csv("rnd2.txt", sep="\t", header=None, index=False)

In [ ]:
mat1 = spark.sparkContext.textFile("rnd1.txt")

## Conversion d'une matrice au format Spark

Lorsqu'un traitement est distribu? en Map/Reduce, il n'est pas possible de s'appuyer sur l'ordre dans lequel sont trait?es les lignes. Le plus est d'ajouter cette information sur chaque ligne plut?t que de chercher ? la r?cup?rer.

In [ ]:
df1.to_csv("rnd1.txt", sep="\t", header=None, index=True)
df2.to_csv("rnd2.txt", sep="\t", header=None, index=True)

In [ ]:
def process_mat_row(row):
    values = row.split("\t")
    index = int(values[0])
    values = [float(_) for _ in values[1:]]
    return [[index, j, v] for j, v in enumerate(values)]

In [ ]:
mat1 = sc.textFile("rnd1.txt")
new_mat1 = mat1.flatMap(process_mat_row)
new_mat1.take(12)

In [ ]:
mat2 = sc.textFile("rnd2.txt")
new_mat2 = mat2.flatMap(process_mat_row)
new_mat2.take(12)

## Produit matriciel

Il faut d'abord faire la jointure avec la m?thode [join](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.join). Il faut que la cl? soit sur la premi?re colonne.

In [ ]:
def key_ij(row):
    return row[0], (row[1], row[2])


def key_ji(row):
    return row[1], (row[0], row[2])


mat_join = new_mat1.map(key_ji).join(new_mat2.map(key_ij))
mat_join.take(12)

On effectue le produit matriciel.

In [ ]:
def produit_matriciel(row):
    index, ((i, v1), (j, v2)) = row
    return i, j, v1 * v2


produit = mat_join.map(produit_matriciel)
produit.take(12)

Il ne reste plus qu'? agr?ger [reduceByKey](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.reduceByKey). La documentation fournit un exemple facilement transposable. Elle indique aussi : *Merge the values for each key using an associative and commutative reduce function.* Pourquoi pr?cise-t-elle **associative et commutative** ? Cela signifie que le r?sultat ne d?pend pas de l'ordre dans lequel l'agr?gation est r?alis?e et qu'on peut commencer ? agr?ger sans attendre d'avoir regroup? toutes les valeurs associ?es ? une cl?.

* *Cas 1 :* [groupBy](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.groupBy) + agr?gation qui commence une fois les valeurs regroup?es
* *Cas 2 :* [reduceByKey](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.reduceByKey) + agr?gation qui commence d?s les premi?res valeurs regroup?es

Le cas 2 est moins consommateur en terme de donn?es. Le cas 1 n'est possible que si les valeurs agr?g?es ne sont pas trop nombreuses. Ca tombe bien, dans notre cas, le cas 2 convient.

In [ ]:
from operator import add

final = produit.map(lambda row: ((row[0], row[1]), row[2])).reduceByKey(add)
aslist = final.collect()
aslist.sort()
aslist

R?sultat initial :

In [ ]:
rnd1 @ rnd2

## M?me algorithme avec les Spark DataFrame

On a besoin de r?aliser un [flatMap](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.flatMap). Une fa?on de faire est de cr?er des colonnes qui sont de type compos? : un tableau, une structure. La multiplication des lignes est obtenue avec la fonction [explode](https://spark.apache.org/docs/2.0.2/api/python/pyspark.sql.html#pyspark.sql.functions.explode).

In [ ]:
schema = ["index"] + ["c%d" % i for i in range(1, 11)]
mat1 = spark.createDataFrame(
    pandas.read_csv("rnd1.txt", header=None, sep="\t"), schema=schema
)

In [ ]:
mat1.printSchema()

In [ ]:
schema = ["index"] + ["c%d" % i for i in range(1, 3)]
mat2 = spark.createDataFrame(
    pandas.read_csv("rnd2.txt", header=None, sep="\t"), schema=schema
)

In [ ]:
mat2.printSchema()

Nous allons avoir besoin de quelques-uns des fonctions et types suivant :

* [explode](https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.functions.explode), [posexplode](https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.functions.posexplode), [array](https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.functions.array), [alias](https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.DataFrame.alias)
* [StructType](https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.types.StructType), [StructField](https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.types.StructField)
* [ArrayType](https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.types.ArrayType)
* [DoubleType](https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.types.DoubleType), [IntegerType](https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.types.IntegerType)

Je recommande le type [FloatType](https://spark.apache.org/docs/2.0.2/api/python/pyspark.sql.html#pyspark.sql.types.FloatType) qui prend deux fois moins de place pour une pr?cision moindre mais suffisante dans la plupart des cas.

In [ ]:
from pyspark.sql.types import (
    ArrayType,
    StructField,
    StructType,
    DoubleType,
    IntegerType,
)
from pyspark.sql.functions import explode, posexplode, array
from pyspark.sql import Row

In [ ]:
cols = ["c%d" % i for i in range(1, 11)]
mat1_array = mat1.select(mat1.index, array(*cols).alias("x"))
mat1_array.printSchema()

In [ ]:
mat1_exploded = mat1_array.select("index", posexplode("x"))
mat1_exploded.printSchema()

In [ ]:
mat1.toPandas().shape, mat1_exploded.toPandas().shape

On recommence le m?me proc?d? pour l'autre matrice.

In [ ]:
cols = ["c%d" % i for i in range(1, 3)]
mat2_array = mat2.select(mat2.index, array(*cols).alias("x"))
mat2_exploded = mat2_array.select("index", posexplode("x"))

Il ne reste plus qu'? faire le produit avec la m?thode [join](https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.DataFrame.join) apr?s avoir renomm? les colonnes avant la jointure pour ?viter les ambigu?t?s.

In [ ]:
mat2_exp2 = (
    mat2_exploded.withColumnRenamed("index", "index2")
    .withColumnRenamed("pos", "pos2")
    .withColumnRenamed("col", "col2")
)
produit = mat1_exploded.join(mat2_exp2, mat1_exploded.pos == mat2_exp2.index2)

In [ ]:
produit.printSchema()

In [ ]:
produit.toPandas().head()

In [ ]:
prod = produit.select(
    produit.index.alias("i"),
    produit.pos2.alias("j"),
    (produit.col * produit.col2).alias("val"),
)
final = prod.groupby("i", "j").sum("val")

In [ ]:
final.printSchema()

In [ ]:
df = final.toPandas()

In [ ]:
df.sort_values(["i", "j"]).head()

In [ ]:
df.shape

## Alternatives

Plut?t que d'avoir un table o? chaque ligne repr?sente trois coefficients, pourrions-nous consid?rer une matrice par bloc de 16x16 ? Que gagnerait-on ?

## Fin

In [ ]:
spark.stop()